# Alpha Connectome, sweep 2: noise check, combined winners, 8 passes, ensembles

What this run answers (about 75-90 min on an A100, after the data download):
1. **Noise:** the same model with 3 seeds shows how much scores move by chance.
2. **Combined winners:** sweep 1's best changes together (mix loss + diff inputs), at widths 128 / 192 / 256.
3. **Longer training:** 8 passes instead of 4 (compare with sweep 1).
4. **Ensembles:** average the best models (chosen on the train holdout) and export the result as one ONNX.
5. **Speed:** time each candidate row by row on this machine's CPU, relative to the official baseline.

Keep the shared `seed` at 0 (default): it fixes the 500-sequence train holdout, so all sweeps are comparable.

**Runtime:** Runtime > Change runtime type > **A100 or L4 GPU** and **High-RAM**.
Results go to Google Drive (`USE_DRIVE = True`), so they survive a disconnect.

Code below is generated from `pipeline/` by `colab/build_notebook.py`; edit there, not here.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "CPUs: $(nproc)"; free -g | head -2; df -h /content | tail -1

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
CPUs: 12
               total        used        free      shared  buff/cache   available
Mem:             167           2         162           0           3         164
overlay         236G   43G  194G  18% /


In [2]:
USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/wunder-connectome/runs' if USE_DRIVE else '/content/runs'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
import os; os.makedirs(OUT_DIR, exist_ok=True); print('results ->', OUT_DIR)

Mounted at /content/drive
results -> /content/drive/MyDrive/wunder-connectome/runs


## 1. Data (33.7 GB; ~10-20 min)
Downloads to a resumable file, unpacks, then checks every parquet file opens with the right number of sequences. Skipped when the data is already there and valid; re-downloaded when anything is broken.

In [3]:
%%bash
mkdir -p /content/wnn && cd /content/wnn
check() {
python3 - <<'EOF'
import sys, pyarrow.parquet as pq
ok = True
for f, n in {'train': 10607, 'valid': 1873, 'valid_mask': 1873}.items():
    try:
        g = pq.ParquetFile(f'datasets/{f}.parquet').metadata.num_row_groups
        print(f'{f}: {g} sequences', 'OK' if g == n else f'(expected {n})'); ok &= g == n
    except Exception as e:
        print(f'{f}: BROKEN ({type(e).__name__}: {e})'); ok = False
sys.exit(0 if ok else 1)
EOF
}
if check; then echo 'data OK, nothing to do'; else
  rm -rf datasets
  URL=https://files.wundernn.io/wnn_connectome_starterpack.tar.gz
  for i in $(seq 10); do   # -C - resumes where a dropped connection stopped
    curl -L -C - --retry 20 --retry-all-errors -sS -o pack.tar.gz "$URL" && break
    echo "download interrupted, resuming ($i)"; sleep 5
  done
  ls -la pack.tar.gz
  tar -xzf pack.tar.gz --strip-components=1 --warning=no-unknown-keyword && find . -name '._*' -delete
  if check; then rm pack.tar.gz; echo 'data OK'; else echo 'DATA STILL BROKEN: delete /content/wnn/pack.tar.gz and re-run this cell'; exit 1; fi
fi
du -sh datasets

train: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/train.parquet'. Detail: [errno 2] No such file or directory)
valid: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid.parquet'. Detail: [errno 2] No such file or directory)
valid_mask: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid_mask.parquet'. Detail: [errno 2] No such file or directory)
-rw-r--r-- 1 root root 33737976954 Oct 23 10:19 pack.tar.gz
train: 10607 sequences OK
valid: 1873 sequences OK
valid_mask: 1873 sequences OK
data OK
33G	datasets


curl: (56) Recv failure: Connection reset by peer
curl: (56) Send failure: Broken pipe
curl: (56) Send failure: Broken pipe
curl: (56) Send failure: Broken pipe
curl: (56) Send failure: Broken pipe


In [4]:
!pip -q install onnx onnxruntime
%cd /content/wnn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.1 MB/s eta 0:00:00
/content/wnn


## 2. Pipeline code

In [5]:
!mkdir -p /content/wnn/pipeline

In [6]:
%%writefile /content/wnn/pipeline/__init__.py

Writing /content/wnn/pipeline/__init__.py


In [7]:
%%writefile /content/wnn/pipeline/data.py
"""Fast sequence loading from the competition parquet files.

Each row group is one 20k-step sequence. Readers keep one ParquetFile per thread
(re-opening parses a large footer) and write straight into a preallocated batch array.
"""
import threading
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pyarrow.parquet as pq

from utils import FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS, WARMUP

N_FEATURES = len(FEATURE_COLUMNS)


class SeqStore:
    def __init__(self, path: str, n_threads: int = 4):
        self.path = path
        pf = pq.ParquetFile(path)
        self.n_seqs = pf.num_row_groups
        self.has_mask = "is_scored" in pf.schema_arrow.names
        self.cols = list(FEATURE_COLUMNS) + list(TARGET_COLUMNS) + (["is_scored"] if self.has_mask else [])
        self._local = threading.local()
        self._pool = ThreadPoolExecutor(n_threads)
        self._cache = None

    def enable_cache(self):
        """Keep every sequence in RAM after its first read (~9 MB each)."""
        self._cache = {}

    def _pf(self):
        if not hasattr(self._local, "pf"):
            self._local.pf = pq.ParquetFile(self.path)
        return self._local.pf

    def _load_into(self, i, g, X, Y, M):
        if self._cache is not None and g in self._cache:
            X[i], Y[i], M[i] = self._cache[g]
            return
        df = self._pf().read_row_group(int(g), columns=self.cols, use_threads=False).to_pandas()
        a = df.to_numpy(np.float32)
        X[i] = a[:, :N_FEATURES]
        Y[i] = a[:, N_FEATURES:N_FEATURES + 2]
        need = np.arange(SEQUENCE_LENGTH) >= WARMUP
        M[i] = need & (a[:, -1] > 0.5) if self.has_mask else need
        if self._cache is not None:
            self._cache[g] = (X[i].copy(), Y[i].copy(), M[i].copy())

    def load(self, ids):
        """-> X [B,T,112] f32, Y [B,T,2] f32, M [B,T] bool (scored rows; all required rows if no mask)."""
        B = len(ids)
        X = np.empty((B, SEQUENCE_LENGTH, N_FEATURES), np.float32)
        Y = np.empty((B, SEQUENCE_LENGTH, 2), np.float32)
        M = np.empty((B, SEQUENCE_LENGTH), bool)
        list(self._pool.map(lambda ig: self._load_into(ig[0], ig[1], X, Y, M), enumerate(ids)))
        return X, Y, M

    def batches(self, ids, batch_size):
        """Yield loaded batches, prefetching the next one in the background."""
        chunks = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]
        if not chunks:
            return
        prefetch = ThreadPoolExecutor(1)
        fut = prefetch.submit(self.load, chunks[0])
        for k in range(len(chunks)):
            batch = fut.result()
            if k + 1 < len(chunks):
                fut = prefetch.submit(self.load, chunks[k + 1])
            yield batch
        prefetch.shutdown()


def split_train(n_seqs: int, n_holdout: int, seed: int = 0):
    """Deterministic split of train row groups into (train_ids, holdout_ids)."""
    perm = np.random.default_rng(seed).permutation(n_seqs)
    return np.sort(perm[n_holdout:]), np.sort(perm[:n_holdout])

Writing /content/wnn/pipeline/data.py


In [8]:
%%writefile /content/wnn/pipeline/model.py
"""Stateful GRU model, metric-aligned losses, ONNX step export and baseline import."""
import numpy as np
import torch
from torch import nn

N_FEATURES = 112
CLIP = 2.0


class GRUNet(nn.Module):
    def __init__(self, hidden=128, layers=2, use_diff=False, dropout=0.0):
        super().__init__()
        self.config = dict(hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout)
        self.use_diff = use_diff
        in_dim = N_FEATURES * (2 if use_diff else 1)
        self.gru = nn.GRU(in_dim, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, 2)

    def forward(self, x, h=None, x_prev=None):
        """x [B,T,112]; h [L,B,H] or None; x_prev [B,112] = last row of the previous chunk
        (None at sequence start, where the first diff is zero)."""
        if self.use_diff:
            first = x[:, :1] if x_prev is None else x_prev[:, None]
            prev = torch.cat([first, x[:, :-1]], 1)
            x = torch.cat([x, x - prev], -1)
        out, h = self.gru(x, h)
        return self.head(out), h


def wmse_loss(p, y):
    """Weighted MSE with the metric's clipping and |y| weights. p, y: [N, 2]."""
    y = y.clamp(-CLIP, CLIP)
    w = y.abs()
    return ((w * (p - y) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()


def wpearson(p, y):
    """Per-target weighted Pearson (pooled over the rows given), metric conventions. -> [2]."""
    y = y.clamp(-CLIP, CLIP)
    w = y.abs()
    W = w.sum(0) + 1e-8
    yc = y - (w * y).sum(0) / W
    pc = p - (w * p).sum(0) / W
    cov = (w * yc * pc).sum(0)
    return cov / torch.sqrt((w * yc * yc).sum(0) * (w * pc * pc).sum(0) + 1e-12)


def compute_loss(kind, p, y, alpha=0.1):
    if kind == "wmse":
        return wmse_loss(p, y)
    if kind == "pearson":
        return 1 - wpearson(p, y).mean()
    if kind == "mix":  # correlation objective, small wmse term anchors the scale for clipping
        return 1 - wpearson(p, y).mean() + alpha * wmse_loss(p, y)
    raise ValueError(kind)


class EnsembleNet(nn.Module):
    """Average of several GRUNets. Each member's output is multiplied by `scales[k]` (per target)
    first, so members with different prediction spreads count equally. h is a list, one per member."""

    def __init__(self, nets, scales):
        super().__init__()
        self.nets = nn.ModuleList(nets)
        self.register_buffer("scales", torch.as_tensor(np.asarray(scales), dtype=torch.float32))  # [K, 2]
        self.use_diff = any(n.use_diff for n in nets)
        self.config = {"members": [n.config for n in nets], "scales": np.asarray(scales).tolist()}

    def forward(self, x, h=None, x_prev=None):
        h = h if h is not None else [None] * len(self.nets)
        preds, new_h = [], []
        for k, (net, hk) in enumerate(zip(self.nets, h)):
            p, hn = net(x, hk, x_prev if net.use_diff else None)
            preds.append(p * self.scales[k])
            new_h.append(hn)
        return torch.stack(preds).mean(0), new_h


class _Step(nn.Module):
    """One-row step graph for ONNX: (x [1,1,112], x_prev [1,112], h_0.. [L,1,H]) -> (pred [1,1,2], h_out_0..)."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x, x_prev, *hs):
        xp = x_prev if self.net.use_diff else None
        if isinstance(self.net, EnsembleNet):
            pred, new_h = self.net(x, list(hs), xp)
            return (pred, *new_h)
        return self.net(x, hs[0], xp)


def export_onnx(net, path: str):
    """GRUNet -> inputs (x, x_prev, h); EnsembleNet -> inputs (x, x_prev, h_0, h_1, ...).
    x_prev is dropped from the graph when no model uses diff features."""
    net = net.eval().cpu()
    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]
    hs = [torch.zeros(m.config["layers"], 1, m.config["hidden"]) for m in members]
    single = not isinstance(net, EnsembleNet)
    h_in = ["h"] if single else [f"h_{k}" for k in range(len(hs))]
    h_out = ["h_out"] if single else [f"h_out_{k}" for k in range(len(hs))]
    args = (torch.zeros(1, 1, N_FEATURES), torch.zeros(1, N_FEATURES), *hs)
    with torch.no_grad():
        torch.onnx.export(_Step(net), args, path, input_names=["x", "x_prev", *h_in],
                          output_names=["pred", *h_out], opset_version=17, dynamo=False)


def load_baseline(onnx_path: str) -> GRUNet:
    """Load the organisers' 2-layer GRU ONNX weights into a GRUNet (for checks / warm starts)."""
    import onnx
    from onnx import numpy_helper

    g = onnx.load(onnx_path).graph
    init = {t.name: numpy_helper.to_array(t) for t in g.initializer}
    grus = [n for n in g.node if n.op_type == "GRU"]
    matmul = next(n for n in g.node if n.op_type == "MatMul")
    H = 128
    net = GRUNet(hidden=H, layers=len(grus))

    def reorder(a):  # ONNX gates z,r,h -> PyTorch r,z,n
        return np.concatenate([a[H:2 * H], a[:H], a[2 * H:]], 0)

    sd = {}
    for l, n in enumerate(grus):
        W, R, B = (init[n.input[i]][0] for i in (1, 2, 3))
        sd[f"gru.weight_ih_l{l}"] = reorder(W)
        sd[f"gru.weight_hh_l{l}"] = reorder(R)
        sd[f"gru.bias_ih_l{l}"] = reorder(B[:3 * H])
        sd[f"gru.bias_hh_l{l}"] = reorder(B[3 * H:])
    sd["head.weight"] = init[matmul.input[1]].T
    sd["head.bias"] = init["reg_head.bias"]
    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in sd.items()})
    return net

Writing /content/wnn/pipeline/model.py


In [9]:
%%writefile /content/wnn/pipeline/fast_eval.py
"""Batched evaluation: runs whole sequences through the network at once (seconds instead of
the ~15 min row-by-row loop) and scores them with the official GlobalAccumulator.

    python -m pipeline.fast_eval --baseline                 # should reproduce 0.617052
    python -m pipeline.fast_eval --ckpt runs/NAME/best.pt --holdout 500
"""
import argparse
import json
import time

import numpy as np
import torch

from utils import SEQUENCE_LENGTH, WARMUP, GlobalAccumulator

from .data import SeqStore, split_train
from .model import GRUNet, load_baseline

NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP


def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def predict_batch(net, X, device, chunk=2000):
    B, T, _ = X.shape
    out = np.empty((B, T, 2), np.float32)
    h = xp = None
    for s in range(0, T, chunk):
        x = torch.from_numpy(X[:, s:s + chunk]).to(device)
        p, h = net(x, h, xp)
        xp = x[:, -1]
        out[:, s:s + chunk] = p.float().cpu().numpy()
    return out


def evaluate_many(nets, store: SeqStore, ids, device, batch_size=128):
    """Score several models in one pass over the data. Per model:
    {'scored': WP on scored rows, 'all': WP on all required rows, ...} (equal without a mask)."""
    for net in nets:
        net.eval()
    accs = [(GlobalAccumulator(), GlobalAccumulator()) for _ in nets]
    for X, Y, M in store.batches(list(ids), batch_size):
        for net, (acc, acc_all) in zip(nets, accs):
            P = predict_batch(net, X, device)
            for b in range(len(X)):
                acc.add(Y[b], P[b], M[b])
                if store.has_mask:
                    acc_all.add(Y[b], P[b], NEED)
    out = []
    for acc, acc_all in accs:
        r = acc.result()
        res = {"scored": r["weighted_pearson"], "t0": r["t0"], "t1": r["t1"], "n_seqs": r["blocks"]}
        res["all"] = acc_all.result()["weighted_pearson"] if store.has_mask else res["scored"]
        out.append(res)
    return out


def evaluate(net, store: SeqStore, ids, device, batch_size=128):
    return evaluate_many([net], store, ids, device, batch_size)[0]


def load_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    net = GRUNet(**ck["config"])
    net.load_state_dict(ck["state_dict"])
    return net


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--baseline", action="store_true")
    g.add_argument("--ckpt")
    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")
    ap.add_argument("--holdout", type=int, default=0, help="also score the N-seq train holdout")
    ap.add_argument("--device", default="auto")
    args = ap.parse_args()

    device = pick_device(args.device)
    net = (load_baseline("baseline/baseline.onnx") if args.baseline else load_ckpt(args.ckpt)).to(device)
    out = {}
    t = time.time()
    valid = SeqStore("datasets/valid.parquet")
    out["valid"] = evaluate(net, valid, range(args.valid_seqs or valid.n_seqs), device)
    if args.holdout:
        train = SeqStore("datasets/train.parquet")
        _, hold = split_train(train.n_seqs, args.holdout)
        out["holdout"] = evaluate(net, train, hold, device)
    out["seconds"] = round(time.time() - t, 1)
    print(json.dumps(out, indent=1))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/fast_eval.py


In [10]:
%%writefile /content/wnn/pipeline/trainer.py
"""Shared training loop: one data stream feeds any number of experiments.

Every batch of whole sequences is read from parquet once and then used to train each model in
turn (truncated BPTT, hidden state carried between `chunk`-step pieces, loss on steps >= 99).
Evaluation is shared the same way: one pass over the eval data scores all models.
"""
import json
import math
import os
import shutil
import time
from dataclasses import asdict, dataclass, field

import numpy as np
import torch

from utils import WARMUP

from .data import SeqStore, split_train
from .fast_eval import evaluate_many
from .model import GRUNet, compute_loss, export_onnx, load_baseline

HERE = os.path.dirname(os.path.abspath(__file__))


@dataclass
class ExpConfig:
    """Per-model settings. Everything about the data stream is shared (see Common)."""
    name: str
    hidden: int = 128
    layers: int = 2
    diff: bool = False
    dropout: float = 0.0
    init_baseline: bool = False
    loss: str = "wmse"          # wmse | pearson | mix
    alpha: float = 0.1          # wmse weight in mix
    lr: float = 2e-3
    wd: float = 1e-4
    seed: int = 0


@dataclass
class Common:
    epochs: int = 4
    batch: int = 256            # sequences per batch
    chunk: int = 500            # TBPTT length
    clip_grad: float = 1.0
    train_seqs: int | None = None
    holdout: int = 500
    valid_seqs: int = 400
    evals_per_epoch: int = 2
    seed: int = 0               # data order + holdout split
    device: str = "auto"
    threads: int = 4            # parquet reader threads
    cache_eval: bool = False    # keep eval data in RAM (~9 MB per sequence)
    cache_train: bool = False   # keep training sequences in RAM after first read
    data_dir: str = "datasets"
    out_dir: str = "runs"
    extra: dict = field(default_factory=dict)


class Experiment:
    def __init__(self, cfg: ExpConfig, common: Common, device, total_steps):
        self.cfg = cfg
        torch.manual_seed(cfg.seed)
        if cfg.init_baseline:
            assert not cfg.diff and cfg.hidden == 128 and cfg.layers == 2, "baseline shape is fixed"
            self.net = load_baseline("baseline/baseline.onnx")
        else:
            self.net = GRUNet(cfg.hidden, cfg.layers, cfg.diff, cfg.dropout)
        self.net.to(device)
        self.opt = torch.optim.AdamW(self.net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        self.sched = torch.optim.lr_scheduler.OneCycleLR(self.opt, max_lr=cfg.lr, total_steps=total_steps,
                                                         pct_start=0.05)
        self.dir = os.path.join(common.out_dir, cfg.name)
        os.makedirs(self.dir, exist_ok=True)
        self.log_f = open(os.path.join(self.dir, "log.jsonl"), "a")
        self.best = -1.0
        self.params = sum(p.numel() for p in self.net.parameters())

    def log(self, **kw):
        line = json.dumps(kw)
        self.log_f.write(line + "\n")
        self.log_f.flush()
        print(f"[{self.cfg.name}] {line}", flush=True)


def _store(common, name):
    return SeqStore(os.path.join(common.data_dir, f"{name}.parquet"), n_threads=common.threads)


def train_many(configs: list[ExpConfig], common: Common):
    from .fast_eval import pick_device

    device = pick_device(common.device)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    rng = np.random.default_rng(common.seed)
    train, valid = _store(common, "train"), _store(common, "valid")
    if common.cache_train:
        train.enable_cache()
    if common.cache_eval:
        valid.enable_cache()
    train_ids, hold_ids = split_train(train.n_seqs, common.holdout, common.seed)
    if common.train_seqs:
        train_ids = rng.choice(train_ids, common.train_seqs, replace=False)
    valid_ids = list(range(min(common.valid_seqs, valid.n_seqs)))
    hold_store = train
    if common.cache_eval and not common.cache_train:  # cache holdout without caching all of train
        hold_store = _store(common, "train")
        hold_store.enable_cache()

    n_batches = math.ceil(len(train_ids) / common.batch)
    total_steps = common.epochs * n_batches * math.ceil(20000 / common.chunk)
    exps = [Experiment(c, common, device, total_steps) for c in configs]
    t0 = time.time()
    for e in exps:
        e.log(event="start", config=asdict(e.cfg), common=asdict(common), device=str(device), params=e.params,
              train_seqs=len(train_ids), holdout=len(hold_ids), steps=total_steps)
    eval_at = set(np.linspace(0, n_batches, common.evals_per_epoch + 1, dtype=int)[1:] - 1)

    def run_eval(epoch, bi):
        nets = [e.net for e in exps]
        vs = evaluate_many(nets, valid, valid_ids, device)
        hs = evaluate_many(nets, hold_store, hold_ids, device)
        for e, v, h in zip(exps, vs, hs):
            sel = (v["scored"] + h["all"]) / 2  # don't select on the visible valid mask alone
            is_best = sel > e.best
            if is_best:
                e.best = sel
                torch.save({"config": e.net.config, "state_dict": e.net.state_dict(), "exp": asdict(e.cfg)},
                           os.path.join(e.dir, "best.pt"))
            e.log(event="eval", epoch=epoch, batch=bi, valid=round(v["scored"], 6), valid_all=round(v["all"], 6),
                  holdout=round(h["all"], 6), select=round(sel, 6), best=is_best, time=round(time.time() - t0, 1))
            e.net.train()

    for epoch in range(common.epochs):
        for e in exps:
            e.net.train()
        ids = list(rng.permutation(train_ids))
        for bi, (X, Y, _) in enumerate(train.batches(ids, common.batch)):
            tb = time.time()
            Xd = torch.from_numpy(X).to(device, non_blocking=True)  # one host->device copy per batch
            Yd = torch.from_numpy(Y).to(device, non_blocking=True)
            for e in exps:
                h = xp = None
                losses = []
                for s in range(0, X.shape[1], common.chunk):
                    x, y = Xd[:, s:s + common.chunk], Yd[:, s:s + common.chunk]
                    p, h = e.net(x, h, xp)
                    lo = max(0, WARMUP - s)  # skip warm-up rows
                    if lo < x.shape[1]:
                        loss = compute_loss(e.cfg.loss, p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2),
                                            e.cfg.alpha)
                        e.opt.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(e.net.parameters(), common.clip_grad)
                        e.opt.step()
                        e.sched.step()
                        losses.append(loss.item())
                    h = h.detach()
                    xp = x[:, -1]
                e.log(event="batch", epoch=epoch, batch=bi, loss=round(float(np.mean(losses)), 5),
                      lr=round(e.sched.get_last_lr()[0], 6))
            print(f"batch {bi + 1}/{n_batches} epoch {epoch + 1}/{common.epochs}: {time.time() - tb:.1f}s "
                  f"for {len(exps)} models, elapsed {(time.time() - t0) / 60:.1f} min", flush=True)
            del Xd, Yd
            if bi in eval_at:
                run_eval(epoch, bi)

    # final: best checkpoints -> full validation + holdout, ONNX submission folders, summary table
    for e in exps:
        ck = torch.load(os.path.join(e.dir, "best.pt"), map_location="cpu", weights_only=False)
        e.net.load_state_dict(ck["state_dict"])
        e.net.to(device)
    nets = [e.net for e in exps]
    full_valid = evaluate_many(nets, _store(common, "valid"), range(valid.n_seqs), device)  # uncached: 17 GB
    full_hold = evaluate_many(nets, hold_store, hold_ids, device)
    rows = []
    for e, v, h in zip(exps, full_valid, full_hold):
        sub = os.path.join(e.dir, "submission")
        os.makedirs(sub, exist_ok=True)
        export_onnx(e.net, os.path.join(sub, "model.onnx"))
        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(sub, "solution.py"))
        res = {"valid_full": v, "holdout": h}
        json.dump(res, open(os.path.join(e.dir, "results.json"), "w"), indent=1)
        e.log(event="done", **res)
        rows.append({"name": e.cfg.name, "params": e.params, "valid_scored": round(v["scored"], 6),
                     "valid_all": round(v["all"], 6), "holdout_all": round(h["all"], 6),
                     "t0": round(v["t0"], 4), "t1": round(v["t1"], 4)})
    rows.sort(key=lambda r: -(r["valid_scored"] + r["holdout_all"]))
    with open(os.path.join(common.out_dir, "summary.jsonl"), "a") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print("\n" + " | ".join(rows[0].keys()))
    for r in rows:
        print(" | ".join(str(v) for v in r.values()))
    return rows

Writing /content/wnn/pipeline/trainer.py


In [11]:
%%writefile /content/wnn/pipeline/multi_train.py
"""Train several experiments side by side from one data stream.

    python -m pipeline.multi_train --config experiments.json --epochs 4 --threads 8 --cache-eval

experiments.json is a list of ExpConfig dicts, e.g.
    [{"name": "wmse"}, {"name": "mix", "loss": "mix"}, {"name": "diff_h192", "diff": true, "hidden": 192}]
Shared settings (epochs, batch, chunk, data/out dirs, ...) are the Common fields, as flags.
"""
import argparse
import dataclasses
import json

from .trainer import Common, ExpConfig, train_many


def add_common_flags(ap):
    for f in dataclasses.fields(Common):
        if f.name == "extra":
            continue
        flag = "--" + f.name.replace("_", "-")
        if f.type is bool:
            ap.add_argument(flag, action="store_true")
        else:  # train_seqs defaults to None (= all) but takes an int
            ap.add_argument(flag, type=int if f.default is None else type(f.default), default=f.default)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True, help="JSON list of experiment configs")
    add_common_flags(ap)
    args = vars(ap.parse_args())
    configs = [ExpConfig(**c) for c in json.load(open(args.pop("config")))]
    names = [c.name for c in configs]
    assert len(set(names)) == len(names), "experiment names must be unique"
    train_many(configs, Common(**args))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/multi_train.py


In [12]:
%%writefile /content/wnn/pipeline/train.py
"""Train one stateful GRU (thin wrapper over trainer.train_many; see multi_train.py for several).

    python -m pipeline.train --name quick --train-seqs 1000 --epochs 2
    python -m pipeline.train --name gru128_wmse --epochs 4

Outputs in runs/NAME/: best.pt, log.jsonl, results.json, submission/{solution.py, model.onnx}.
"""
import argparse
import dataclasses

from .multi_train import add_common_flags
from .trainer import Common, ExpConfig, train_many


def main():
    ap = argparse.ArgumentParser()
    exp_fields = {f.name: f for f in dataclasses.fields(ExpConfig)}
    for name, f in exp_fields.items():
        flag = "--" + name.replace("_", "-")
        if name == "name":
            ap.add_argument(flag, required=True)
        elif f.type is bool:
            ap.add_argument(flag, action="store_true")
        else:
            ap.add_argument(flag, type=type(f.default), default=f.default)
    add_common_flags(ap)
    args = vars(ap.parse_args())
    exp = ExpConfig(**{k: args.pop(k) for k in list(args) if k in exp_fields})
    train_many([exp], Common(**args))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/train.py


In [13]:
%%writefile /content/wnn/pipeline/ensemble.py
"""Score models alone and averaged, pick an ensemble on the train holdout, optionally export it.

    python -m pipeline.ensemble --ckpts runs/*/best.pt [baseline] --export runs/ens_best/submission

Every model is run once over the holdout and validation data (shared loading); combinations are then
scored from the stored predictions. Members are rescaled to a common prediction spread before
averaging (scales measured on the holdout). Selection uses the holdout only, so the validation
numbers stay an honest check of the chosen combination.
"""
import argparse
import glob
import json
import os
import shutil
import time

import numpy as np

from utils import SEQUENCE_LENGTH, WARMUP, weighted_pearson

from .data import SeqStore, split_train
from .fast_eval import load_ckpt, pick_device, predict_batch
from .model import EnsembleNet, export_onnx, load_baseline

HERE = os.path.dirname(os.path.abspath(__file__))
NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP


def load_member(path):
    if path == "baseline":
        return "baseline", load_baseline("baseline/baseline.onnx")
    return os.path.basename(os.path.dirname(path)), load_ckpt(path)


def collect(nets, store, ids, device, masks):
    """-> Y {mask: [N,2]}, P {mask: [K,N,2]} for masks 'all' (required rows) and/or 'scored'."""
    ys = {m: [] for m in masks}
    ps = {m: [[] for _ in nets] for m in masks}
    for X, Y, M in store.batches(list(ids), 128):
        sel = {"all": np.broadcast_to(NEED, M.shape), "scored": M}
        for m in masks:
            ys[m].append(Y[sel[m]])
        for k, net in enumerate(nets):
            P = predict_batch(net, X, device)
            for m in masks:
                ps[m][k].append(P[sel[m]])
    return ({m: np.concatenate(ys[m]) for m in masks},
            {m: np.stack([np.concatenate(p) for p in ps[m]]) for m in masks})


def wp(y, p):
    return float(np.mean([weighted_pearson(y[:, j], p[:, j]) for j in range(2)]))


def combine(P, scales, idx):
    return (P[idx] * scales[idx][:, None, :]).mean(0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpts", nargs="+", required=True, help="best.pt paths/globs, or 'baseline'")
    ap.add_argument("--holdout", type=int, default=500)
    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")
    ap.add_argument("--max-members", type=int, default=5)
    ap.add_argument("--export", help="write the greedy ensemble as a submission folder here")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args()

    paths = []
    for c in args.ckpts:
        paths += [c] if c == "baseline" else sorted(glob.glob(c))
    names, nets = zip(*[load_member(p) for p in paths])
    device = pick_device(args.device)
    nets = [n.eval().to(device) for n in nets]
    print(f"{len(nets)} models: {', '.join(names)}", flush=True)

    t = time.time()
    train = SeqStore("datasets/train.parquet", args.threads)
    valid = SeqStore("datasets/valid.parquet", args.threads)
    _, hold_ids = split_train(train.n_seqs, args.holdout)
    Yh, Ph = collect(nets, train, hold_ids, device, ["all"])
    Yv, Pv = collect(nets, valid, range(args.valid_seqs or valid.n_seqs), device, ["scored", "all"])
    print(f"predictions collected in {time.time() - t:.0f}s", flush=True)

    sd = Ph["all"].std(axis=1)  # [K, 2]
    scales = sd.mean(0, keepdims=True) / sd

    def score(idx):
        return {"holdout": wp(Yh["all"], combine(Ph["all"], scales, idx)),
                "valid_scored": wp(Yv["scored"], combine(Pv["scored"], scales, idx)),
                "valid_all": wp(Yv["all"], combine(Pv["all"], scales, idx))}

    rows = []
    singles = [(k, score([k])) for k in range(len(nets))]
    for k, s in singles:
        rows.append({"combo": names[k], "n": 1, **s})
    order = [k for k, s in sorted(singles, key=lambda ks: -ks[1]["holdout"])]
    for top in range(2, min(len(nets), args.max_members) + 1):
        rows.append({"combo": f"top{top} by holdout", "n": top, "members": [names[k] for k in order[:top]],
                     **score(order[:top])})

    # greedy forward selection on the holdout (Caruana-style, no repeats)
    chosen, best = [order[0]], singles[order[0]][1]["holdout"]
    while len(chosen) < args.max_members:
        cand = [(k, wp(Yh["all"], combine(Ph["all"], scales, chosen + [k]))) for k in range(len(nets)) if k not in chosen]
        if not cand:
            break
        k, s = max(cand, key=lambda c: c[1])
        if s <= best:
            break
        chosen.append(k)
        best = s
    rows.append({"combo": "greedy (holdout)", "n": len(chosen), "members": [names[k] for k in chosen],
                 **score(chosen)})

    print(f"\n{'combo':34s} {'n':>2s} {'holdout':>8s} {'valid_sc':>8s} {'valid_all':>9s}")
    for r in rows:
        print(f"{r['combo'][:34]:34s} {r['n']:2d} {r['holdout']:8.5f} {r['valid_scored']:8.5f} {r['valid_all']:9.5f}"
              + (f"   {'+'.join(r['members'])}" if "members" in r else ""))

    if args.export:
        os.makedirs(args.export, exist_ok=True)
        if len(chosen) == 1:
            net = nets[chosen[0]]
        else:
            net = EnsembleNet([nets[k] for k in chosen], scales[chosen])
        export_onnx(net, os.path.join(args.export, "model.onnx"))
        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(args.export, "solution.py"))
        json.dump({"members": [paths[k] for k in chosen], "scales": scales[chosen].tolist(), "scores": rows[-1]},
                  open(os.path.join(args.export, "ensemble.json"), "w"), indent=1)
        print(f"\nexported {len(chosen)}-model ensemble -> {args.export}")


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/ensemble.py


In [14]:
%%writefile /content/wnn/pipeline/speed.py
"""Time submission folders row by row on one CPU thread, next to the official baseline.

    python -m pipeline.speed runs/NAME/submission runs/ens/submission

The test set is 39.4M rows in 60 min (~91 us/row on the grader's CPU). The baseline is known to
fit, so the ratio to the baseline is the number to watch; absolute times depend on this machine.
"""
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"

import argparse
import importlib.util
import sys
import time

import numpy as np
import pyarrow.parquet as pq

from utils import FEATURE_COLUMNS, WARMUP, DataPoint

TEST_ROWS = 39_400_000


def load_model(folder, i):
    folder = os.path.abspath(folder)
    sys.path.insert(0, folder)
    spec = importlib.util.spec_from_file_location(f"solution_{i}", os.path.join(folder, "solution.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.PredictionModel()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folders", nargs="+")
    ap.add_argument("--rows", type=int, default=20000)
    args = ap.parse_args()

    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, columns=list(FEATURE_COLUMNS))
    X = np.column_stack([tbl[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)[:args.rows]

    results = []
    for i, folder in enumerate(["baseline"] + args.folders):
        model = load_model(folder, i)
        t = time.perf_counter()
        for step in range(len(X)):
            model.predict(DataPoint(0, step, step >= WARMUP, X[step]))
        us = (time.perf_counter() - t) / len(X) * 1e6
        results.append((folder, us))
        base = results[0][1]
        print(f"{folder:50s} {us:7.1f} us/row  {us / base:5.2f}x baseline  "
              f"~{us * TEST_ROWS / 60e6:5.1f} min for the test set here", flush=True)


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/speed.py


In [15]:
%%writefile /content/wnn/pipeline/solution_template.py
"""Generic stateful ONNX step runner. Copied as solution.py next to model.onnx by the pipeline.
Works for single models (input "h") and ensembles (inputs "h_0", "h_1", ...)."""
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import onnxruntime as ort


class PredictionModel:
    def __init__(self):
        o = ort.SessionOptions()
        o.intra_op_num_threads = 1
        o.inter_op_num_threads = 1
        o.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        o.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        o.add_session_config_entry("session.intra_op.allow_spinning", "0")
        self.session = ort.InferenceSession(str(Path(__file__).resolve().parent / "model.onnx"),
                                            sess_options=o, providers=["CPUExecutionProvider"])
        inputs = self.session.get_inputs()
        # hidden-state inputs, in graph order; outputs[1:] are their updates in the same order
        self.h_names = [i.name for i in inputs if i.name not in ("x", "x_prev")]
        self.h_init = [np.zeros(i.shape, dtype=np.float32) for i in inputs if i.name in self.h_names]
        self.use_prev = any(i.name == "x_prev" for i in inputs)  # dropped when no model uses diff features
        self.seq_ix = None

    def predict(self, data_point):
        if data_point.seq_ix != self.seq_ix:
            self.seq_ix = data_point.seq_ix
            self.h = [h.copy() for h in self.h_init]
            self.x_prev = None
        x = np.asarray(data_point.state, dtype=np.float32).reshape(1, 1, 112)
        if not np.isfinite(x).all():  # a NaN would otherwise stay in the hidden state for the whole sequence
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        feeds = dict(zip(self.h_names, self.h))
        feeds["x"] = x
        if self.use_prev:
            feeds["x_prev"] = x[0] if self.x_prev is None else self.x_prev
        out = self.session.run(None, feeds)
        self.h = out[1:]
        self.x_prev = x[0]
        if not data_point.need_prediction:
            return None
        pred = out[0].reshape(2)
        if not np.isfinite(pred).all():
            pred = np.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)
        return pred

Writing /content/wnn/pipeline/solution_template.py


## 3. Sanity check: baseline weights through our fast evaluator (300 seqs: 0.6246; full valid: 0.617052)

In [16]:
!python -m pipeline.fast_eval --baseline --valid-seqs 300

{
 "valid": {
  "scored": 0.6246286498889674,
  "t0": 0.6038379686442771,
  "t1": 0.6454193311336577,
  "n_seqs": 300,
  "all": 0.4380707773677913
 },
 "seconds": 4.9
}


## 4. Experiments

In [17]:
import json
experiments = [{'name': 's0_wmse_h128', 'seed': 0},
 {'name': 's1_wmse_h128', 'seed': 1},
 {'name': 's2_wmse_h128', 'seed': 2},
 {'name': 'c1_mix_diff_h128', 'loss': 'mix', 'diff': True},
 {'name': 'c2_mix_diff_h128_drop', 'loss': 'mix', 'diff': True, 'dropout': 0.1},
 {'name': 'c3_mix_diff_h192_drop', 'loss': 'mix', 'diff': True, 'hidden': 192, 'dropout': 0.1},
 {'name': 'c3b_mix_diff_h192_drop_s1', 'loss': 'mix', 'diff': True, 'hidden': 192, 'dropout': 0.1, 'seed': 1},
 {'name': 'c4_mix_diff_h256_drop', 'loss': 'mix', 'diff': True, 'hidden': 256, 'dropout': 0.1}]
json.dump(experiments, open('experiments.json', 'w'), indent=1)
print(len(experiments), 'experiments')

8 experiments


## 5. Train (8 passes, one evaluation per pass)

In [18]:
SWEEP = f'{OUT_DIR}/sweep2'
!python -m pipeline.multi_train --config experiments.json --epochs 8 --evals-per-epoch 1 --threads 8 --cache-eval --out-dir {SWEEP} 2>&1 | grep -v '"event": "batch"'

[s0_wmse_h128] {"event": "start", "config": {"name": "s0_wmse_h128", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha": 0.1, "lr": 0.002, "wd": 0.0001, "seed": 0}, "common": {"epochs": 8, "batch": 256, "chunk": 500, "clip_grad": 1.0, "train_seqs": null, "holdout": 500, "valid_seqs": 400, "evals_per_epoch": 1, "seed": 0, "device": "auto", "threads": 8, "cache_eval": true, "cache_train": false, "data_dir": "datasets", "out_dir": "/content/drive/MyDrive/wunder-connectome/runs/sweep2", "extra": {}}, "device": "cuda", "params": 192258, "train_seqs": 10107, "holdout": 500, "steps": 12800}
[s1_wmse_h128] {"event": "start", "config": {"name": "s1_wmse_h128", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha": 0.1, "lr": 0.002, "wd": 0.0001, "seed": 1}, "common": {"epochs": 8, "batch": 256, "chunk": 500, "clip_grad": 1.0, "train_seqs": null, "holdout": 500, "valid_seqs": 400, "evals_

## 6. Results

In [19]:
import pandas as pd
rows = [json.loads(l) for l in open(f'{SWEEP}/summary.jsonl')]
df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
df['select'] = (df.valid_scored + df.holdout_all) / 2
df.sort_values('select', ascending=False)

,name,params,valid_scored,valid_all,holdout_all,t0,t1,select
0,c1_mix_diff_h128,235266,0.677716,0.505692,0.451784,0.6733,0.6822,0.564750
1,s2_wmse_h128,192258,0.677971,0.505136,0.449955,0.6734,0.6825,0.563963
2,c4_mix_diff_h256_drop,765442,0.676898,0.505106,0.449754,0.6718,0.6820,0.563326
3,s1_wmse_h128,192258,0.677324,0.505270,0.448793,0.6732,0.6814,0.563059
4,c2_mix_diff_h128_drop,235266,0.676768,0.505073,0.449185,0.6725,0.6810,0.562976
5,c3_mix_diff_h192_drop,463490,0.677472,0.505259,0.447208,0.6732,0.6817,0.562340
6,c3b_mix_diff_h192_drop_s1,463490,0.676675,0.504366,0.447371,0.6713,0.6820,0.562023
7,s0_wmse_h128,192258,0.677361,0.503390,0.444888,0.6722,0.6825,0.561125


## 7. Ensembles
Uses the sweep-2 models plus sweep 1's useful ones (pure-Pearson e03 left out). The official baseline is left out
because it was probably trained on our holdout sequences, so its holdout score is not an honest one.
`MAX_MEMBERS` bounds inference cost: each 128-unit model costs ~1x baseline time, 192-unit ~2x, 256-unit ~3.5x.

In [20]:
MAX_MEMBERS = 4
!python -m pipeline.ensemble --ckpts "{SWEEP}/*/best.pt" "{OUT_DIR}/e0[12456]*/best.pt" --threads 8 --max-members {MAX_MEMBERS} --export {OUT_DIR}/ensemble_sweep2/submission 2>&1 | grep -vE 'Warning|_generic_rnn'

13 models: c1_mix_diff_h128, c2_mix_diff_h128_drop, c3_mix_diff_h192_drop, c3b_mix_diff_h192_drop_s1, c4_mix_diff_h256_drop, s0_wmse_h128, s1_wmse_h128, s2_wmse_h128, e01_wmse_h128, e02_mix_h128, e04_wmse_diff_h128, e05_wmse_h192_drop, e06_wmse_h128_lr1e-3
predictions collected in 235s

combo                               n  holdout valid_sc valid_all
c1_mix_diff_h128                    1  0.45178  0.67771   0.50569
c2_mix_diff_h128_drop               1  0.44918  0.67677   0.50507
c3_mix_diff_h192_drop               1  0.44720  0.67746   0.50526
c3b_mix_diff_h192_drop_s1           1  0.44737  0.67667   0.50436
c4_mix_diff_h256_drop               1  0.44975  0.67690   0.50511
s0_wmse_h128                        1  0.44489  0.67736   0.50339
s1_wmse_h128                        1  0.44879  0.67732   0.50527
s2_wmse_h128                        1  0.44995  0.67797   0.50514
e01_wmse_h128                       1  0.44714  0.67594   0.50468
e02_mix_h128                        1  0.45005  0.67

## 8. Speed on this CPU (one thread), relative to the official baseline
The grader allows ~91 µs/row; the baseline fits comfortably, so a ratio up to ~2x is very likely safe.
Re-check finalists on the Mac with `evaluate.py` too.

In [21]:
!python -m pipeline.speed {SWEEP}/s0_wmse_h128/submission {SWEEP}/c1_mix_diff_h128/submission {SWEEP}/c3_mix_diff_h192_drop/submission {SWEEP}/c4_mix_diff_h256_drop/submission {OUT_DIR}/ensemble_sweep2/submission

baseline                                              80.2 us/row   1.00x baseline  ~ 52.6 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep2/s0_wmse_h128/submission    92.3 us/row   1.15x baseline  ~ 60.6 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep2/c1_mix_diff_h128/submission   111.1 us/row   1.39x baseline  ~ 73.0 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep2/c3_mix_diff_h192_drop/submission   150.9 us/row   1.88x baseline  ~ 99.1 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep2/c4_mix_diff_h256_drop/submission   202.7 us/row   2.53x baseline  ~133.1 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/ensemble_sweep2/submission   435.6 us/row   5.43x baseline  ~286.0 min for the test set here


Submission folders (solution.py + model.onnx) are in Drive under `wunder-connectome/runs/`: `sweep2/<name>/submission/` and `ensemble_sweep2/submission/`. Zip one with `solution.py` at the root to upload.